# STT FR - Whisper + pyannote monolithique CPU

Ce notebook est autonome cote code: il ne depend pas de fichiers `.py` du projet.

Il peut installer ses dependances Python depuis une cellule `%pip`, puis lancer la transcription en CPU.

Il lit les audios dans `input/`, charge `HF_TOKEN` depuis `.env.local`, et ecrit les sorties dans `outputs/notebook/`.


## 1. Installation des dependances CPU

Execute cette cellule une seule fois si l'environnement Python n'a pas encore les dependances.

Apres installation, redemarre le kernel puis relance le notebook depuis le debut. Le token Hugging Face reste a mettre dans `.env.local` ou dans la variable `HF_TOKEN`.


## 2. Initialisation

Cette cellule retrouve la racine du projet et charge `.env.local` si disponible.


In [1]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
import time
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from pprint import pprint
from typing import Any, Callable

import numpy as np
import soundfile as sf


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "input").exists() or (candidate / ".env.local").exists():
            return candidate
    return current


ROOT = find_project_root(Path.cwd())
INPUT_DIR = ROOT / "input"
OUTPUT_DIR = ROOT / "outputs" / "notebook"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env.local")
except ModuleNotFoundError:
    pass

print("ROOT =", ROOT)
print("INPUT_DIR =", INPUT_DIR)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("HF_TOKEN loaded =", bool(os.environ.get("HF_TOKEN")))

ROOT = /app
INPUT_DIR = /app/input
OUTPUT_DIR = /app/outputs/notebook
HF_TOKEN loaded = True


In [2]:


AUDIO_EXTENSIONS = {".aac", ".flac", ".m4a", ".mp3", ".mp4", ".ogg", ".opus", ".wav", ".webm", ".wma"}


@dataclass(frozen=True)
class RuntimeSettings:
    language: str = "fr"
    whisper_model: str = "large-v3-turbo"
    device: str = "cpu"
    compute_type: str = "int8"
    beam_size: int = 5
    vad_filter: bool = True
    min_silence_duration_ms: int = 500
    pyannote_model: str = "pyannote/speaker-diarization-community-1"
    min_speakers: int | None = None
    max_speakers: int | None = None


def slugify(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", value.strip())
    cleaned = cleaned.strip("._-")
    return cleaned or "audio"


def ffmpeg_exe() -> str:
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return "ffmpeg"


def convert_to_wav(source_path: Path, work_dir: Path, sample_rate: int = 16000) -> Path:
    work_dir.mkdir(parents=True, exist_ok=True)
    wav_path = work_dir / f"{slugify(source_path.stem)}.wav"
    command = [
        ffmpeg_exe(),
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-i",
        str(source_path),
        "-ac",
        "1",
        "-ar",
        str(sample_rate),
        "-vn",
        "-f",
        "wav",
        str(wav_path),
    ]
    subprocess.run(command, check=True)
    return wav_path


def wav_duration_seconds(wav_path: Path) -> float:
    info = sf.info(str(wav_path))
    return float(info.frames) / float(info.samplerate)


def load_wav_mono(wav_path: Path) -> tuple[np.ndarray, int]:
    audio, sample_rate = sf.read(str(wav_path), always_2d=True, dtype="float32")
    if audio.shape[1] > 1:
        audio = np.mean(audio, axis=1)
    else:
        audio = audio[:, 0]
    return audio, int(sample_rate)


def cuda_summary() -> dict[str, Any]:
    try:
        import torch
        available = bool(torch.cuda.is_available())
        return {
            "available": available,
            "device_name": torch.cuda.get_device_name(0) if available else None,
            "torch_cuda": torch.version.cuda,
            "device_count": torch.cuda.device_count() if available else 0,
        }
    except Exception as exc:
        return {"available": False, "error": str(exc)}


def load_whisper_model(settings: RuntimeSettings) -> Any:
    from faster_whisper import WhisperModel
    return WhisperModel(settings.whisper_model, device=settings.device, compute_type=settings.compute_type)


def load_pyannote_pipeline(settings: RuntimeSettings, hf_token: str | None) -> Any:
    if not hf_token:
        raise RuntimeError("HF_TOKEN is required for pyannote diarization.")
    import torch
    from pyannote.audio import Pipeline
    pipeline = Pipeline.from_pretrained(settings.pyannote_model, token=hf_token)
    if settings.device == "cuda" and torch.cuda.is_available():
        pipeline.to(torch.device("cuda"))
    return pipeline


def transcribe_whisper(
    model: Any,
    wav_path: Path,
    settings: RuntimeSettings,
    progress: Callable[[str], None] | None = None,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    progress = progress or (lambda _message: None)
    progress("Transcription Whisper en cours")
    vad_parameters = {"min_silence_duration_ms": settings.min_silence_duration_ms}
    segment_iter, info = model.transcribe(
        str(wav_path),
        language=settings.language,
        beam_size=settings.beam_size,
        vad_filter=settings.vad_filter,
        vad_parameters=vad_parameters,
        word_timestamps=True,
        condition_on_previous_text=False,
    )
    segments: list[dict[str, Any]] = []
    for index, segment in enumerate(segment_iter):
        words = []
        for word in segment.words or []:
            words.append({
                "start": float(word.start),
                "end": float(word.end),
                "word": word.word.strip(),
                "speaker": "UNKNOWN",
                "confidence": float(word.probability) if word.probability is not None else None,
            })
        segments.append({
            "id": index,
            "start": float(segment.start),
            "end": float(segment.end),
            "speaker": "UNKNOWN",
            "text": segment.text.strip(),
            "words": words,
        })
        if index and index % 25 == 0:
            progress(f"Transcription Whisper: {index} segments")
    metadata = {
        "language": getattr(info, "language", settings.language),
        "language_probability": getattr(info, "language_probability", None),
        "duration": getattr(info, "duration", None),
    }
    return segments, metadata


def diarize_audio(
    pipeline: Any,
    wav_path: Path,
    settings: RuntimeSettings,
    progress: Callable[[str], None] | None = None,
) -> list[dict[str, Any]]:
    progress = progress or (lambda _message: None)
    progress("Diarisation pyannote en cours")
    import torch
    audio, sample_rate = load_wav_mono(wav_path)
    waveform = torch.from_numpy(audio).float().unsqueeze(0)
    kwargs: dict[str, int] = {}
    if settings.min_speakers is not None:
        kwargs["min_speakers"] = int(settings.min_speakers)
    if settings.max_speakers is not None:
        kwargs["max_speakers"] = int(settings.max_speakers)
    diarization = pipeline({"waveform": waveform, "sample_rate": sample_rate}, **kwargs)
    annotation = (
        getattr(diarization, "exclusive_speaker_diarization", None)
        or getattr(diarization, "speaker_diarization", None)
        or diarization
    )
    turns: list[dict[str, Any]] = []
    for turn, _, speaker in annotation.itertracks(yield_label=True):
        turns.append({"start": float(turn.start), "end": float(turn.end), "speaker": str(speaker)})
    turns.sort(key=lambda item: (item["start"], item["end"], item["speaker"]))
    return turns


def interval_overlap(a_start: float, a_end: float, b_start: float, b_end: float) -> float:
    return max(0.0, min(a_end, b_end) - max(a_start, b_start))


def speaker_for_interval(start: float, end: float, turns: list[dict[str, Any]]) -> str:
    if not turns:
        return "UNKNOWN"
    best_speaker = "UNKNOWN"
    best_overlap = 0.0
    for turn in turns:
        overlap = interval_overlap(start, end, turn["start"], turn["end"])
        if overlap > best_overlap:
            best_overlap = overlap
            best_speaker = turn["speaker"]
    if best_overlap > 0:
        return best_speaker
    midpoint = (start + end) / 2.0
    nearest = min(turns, key=lambda turn: min(abs(midpoint - turn["start"]), abs(midpoint - turn["end"])))
    return nearest["speaker"]


def assign_speakers(segments: list[dict[str, Any]], turns: list[dict[str, Any]]) -> list[dict[str, Any]]:
    for segment in segments:
        word_speakers = []
        for word in segment.get("words") or []:
            speaker = speaker_for_interval(float(word["start"]), float(word["end"]), turns)
            word["speaker"] = speaker
            if speaker != "UNKNOWN":
                word_speakers.append(speaker)
        if word_speakers:
            segment["speaker"] = Counter(word_speakers).most_common(1)[0][0]
        else:
            segment["speaker"] = speaker_for_interval(float(segment["start"]), float(segment["end"]), turns)
    return segments


def timestamp_srt(seconds: float) -> str:
    millis = int(round(seconds * 1000))
    hours, remainder = divmod(millis, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, millis = divmod(remainder, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


def timestamp_vtt(seconds: float) -> str:
    return timestamp_srt(seconds).replace(",", ".")


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
        handle.write("\n")


def write_txt(path: Path, segments: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        current_speaker = None
        for segment in segments:
            speaker = segment.get("speaker") or "UNKNOWN"
            text = (segment.get("text") or "").strip()
            if not text:
                continue
            if speaker != current_speaker:
                if current_speaker is not None:
                    handle.write("\n")
                handle.write(f"[{speaker}]\n")
                current_speaker = speaker
            handle.write(f"{timestamp_vtt(float(segment['start']))} - {timestamp_vtt(float(segment['end']))}  {text}\n")


def write_srt(path: Path, segments: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        index = 1
        for segment in segments:
            text = (segment.get("text") or "").strip()
            if not text:
                continue
            handle.write(f"{index}\n")
            handle.write(f"{timestamp_srt(float(segment['start']))} --> {timestamp_srt(float(segment['end']))}\n")
            handle.write(f"{segment.get('speaker', 'UNKNOWN')}: {text}\n\n")
            index += 1


def write_vtt(path: Path, segments: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        handle.write("WEBVTT\n\n")
        for segment in segments:
            text = (segment.get("text") or "").strip()
            if not text:
                continue
            handle.write(f"{timestamp_vtt(float(segment['start']))} --> {timestamp_vtt(float(segment['end']))}\n")
            handle.write(f"{segment.get('speaker', 'UNKNOWN')}: {text}\n\n")


def write_all_outputs(output_stem: Path, payload: dict[str, Any]) -> dict[str, Path]:
    segments = payload["segments"]
    paths = {
        "json": output_stem.with_suffix(".json"),
        "txt": output_stem.with_suffix(".txt"),
        "srt": output_stem.with_suffix(".srt"),
        "vtt": output_stem.with_suffix(".vtt"),
    }
    write_json(paths["json"], payload)
    write_txt(paths["txt"], segments)
    write_srt(paths["srt"], segments)
    write_vtt(paths["vtt"], segments)
    return paths


def run_pipeline(
    source_path: Path,
    work_dir: Path,
    settings: RuntimeSettings,
    whisper_model: Any,
    pyannote_pipeline: Any,
    progress: Callable[[str], None] | None = None,
) -> dict[str, Any]:
    progress = progress or (lambda _message: None)
    started_at = time.perf_counter()
    prepared_dir = work_dir / "prepared"
    wav_path = convert_to_wav(source_path, prepared_dir)
    duration_seconds = wav_duration_seconds(wav_path)
    segments, whisper_metadata = transcribe_whisper(whisper_model, wav_path, settings, progress)
    diarization_turns = diarize_audio(pyannote_pipeline, wav_path, settings, progress)
    segments = assign_speakers(segments, diarization_turns)
    elapsed_seconds = time.perf_counter() - started_at
    words = [word for segment in segments for word in segment.get("words") or []]
    return {
        "source_audio": str(source_path),
        "pipeline": "faster-whisper+pyannote",
        "language": settings.language,
        "settings": settings.__dict__,
        "runtime": {
            "duration_seconds": duration_seconds,
            "elapsed_seconds": elapsed_seconds,
            "rtf": elapsed_seconds / duration_seconds if duration_seconds else None,
            "cuda": cuda_summary(),
        },
        "whisper": whisper_metadata,
        "segments": segments,
        "diarization_turns": diarization_turns,
        "metrics": {
            "segment_count": len(segments),
            "word_count": len(words),
            "speaker_count": len({segment["speaker"] for segment in segments if segment.get("speaker")}),
            "word_timestamp_coverage": (
                sum(1 for word in words if word.get("start") is not None and word.get("end") is not None) / len(words)
                if words
                else 0.0
            ),
        },
    }

In [3]:
cuda = cuda_summary()
pprint(cuda)

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN missing. Put it in .env.local before running the notebook.")

{'available': False,
 'device_count': 0,
 'device_name': None,
 'torch_cuda': '13.0'}


In [4]:
audio_files = sorted(
    path for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
)

# Optionnel: forcer un fichier precis, par exemple:
# AUDIO_FILE = r"C:\DATA TRAVAIL\transcription\input\mon_audio.m4a"
AUDIO_FILE = None

if AUDIO_FILE:
    audio_path = Path(AUDIO_FILE)
else:
    if not audio_files:
        raise FileNotFoundError(f"No audio file found in {INPUT_DIR}")
    audio_path = audio_files[0]

print("Audio selected:", audio_path)
print("Size MB:", round(audio_path.stat().st_size / 1024 / 1024, 2))

Audio selected: /app/input/Interview_Alexandre_Hocquet_The_Conversation.ogg
Size MB: 16.28


In [5]:
DEVICE = "cpu"
COMPUTE_TYPE = "int8"

settings = RuntimeSettings(
    language="fr",
    whisper_model="large-v3-turbo",
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    beam_size=5,
    vad_filter=True,
    min_silence_duration_ms=500,
    min_speakers=None,
    max_speakers=None,
)

settings

RuntimeSettings(language='fr', whisper_model='large-v3-turbo', device='cpu', compute_type='int8', beam_size=5, vad_filter=True, min_silence_duration_ms=500, pyannote_model='pyannote/speaker-diarization-community-1', min_speakers=None, max_speakers=None)

In [ ]:
whisper_model = load_whisper_model(settings)
diarizer = load_pyannote_pipeline(settings, os.environ.get("HF_TOKEN"))
print("Models loaded")

config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_stem = OUTPUT_DIR / f"{slugify(audio_path.stem)}_{timestamp}"
work_dir = OUTPUT_DIR / "_work" / output_stem.stem

payload = run_pipeline(
    source_path=audio_path,
    work_dir=work_dir,
    settings=settings,
    whisper_model=whisper_model,
    pyannote_pipeline=diarizer,
    progress=print,
)

paths = write_all_outputs(output_stem, payload)
print("Outputs:")
for label, path in paths.items():
    print(label, "->", path)

print("Runtime:")
pprint(payload["runtime"])
print("Metrics:")
pprint(payload["metrics"])

In [ ]:
rows = [
    {
        "start": round(segment["start"], 2),
        "end": round(segment["end"], 2),
        "speaker": segment["speaker"],
        "text": segment["text"],
    }
    for segment in payload["segments"][:30]
]

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ModuleNotFoundError:
    rows

In [ ]:
txt_path = paths["txt"]
print(txt_path)
print(txt_path.read_text(encoding="utf-8")[:4000])

In [ ]:
# ============================================================
# 12. Génération PV + Résumé via Ollama
# ============================================================
import requests
import shutil
 
# --- Paramètres ---
OLLAMA_URL = "http://localhost:11434/api/generate"  # remplace par host.docker.internal si Docker
OLLAMA_MODEL = "qwen3.5:9b"
MAX_WORDS_PER_CHUNK = 4000
 
 
# --- Fonction de base Ollama ---
 
def ollama_generate(prompt: str, model: str = OLLAMA_MODEL) -> str:
    response = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": prompt,
        "stream": False
    })
    response.raise_for_status()
    return response.json()["response"].strip()
 
 
# --- Découpage en chunks ---
 
def split_into_chunks(text: str, max_words: int = MAX_WORDS_PER_CHUNK) -> list[str]:
    """Découpe le texte en chunks en respectant les blocs de locuteurs."""
    chunks = []
    current_chunk = []
    current_word_count = 0
 
    for line in text.split("\n"):
        line_words = len(line.split())
        if current_word_count + line_words > max_words and current_chunk:
            chunks.append("\n".join(current_chunk))
            current_chunk = []
            current_word_count = 0
        current_chunk.append(line)
        current_word_count += line_words
 
    if current_chunk:
        chunks.append("\n".join(current_chunk))
 
    return chunks
 
 
# --- Génération du procès-verbal ---
 
def generate_pv(transcript: str) -> str:
    prompt = f"""Tu es un assistant de réunion. Voici la transcription d'une réunion avec les locuteurs identifiés.
 
TRANSCRIPTION :
{transcript}
 
Réécris cette transcription en procès-verbal en français en respectant ces règles :
- Garde l'ordre chronologique des échanges
- Conserve l'attribution à chaque locuteur (SPEAKER_00, SPEAKER_01...)
- Supprime les répétitions, hésitations et phrases inachevées
- Reformule les phrases maladroites tout en gardant le sens exact
- Ne rajoute rien qui n'a pas été dit
- Ne regroupe pas par thème, reste chronologique
 
Procès-verbal :"""
    return ollama_generate(prompt)
 
 
def generate_pv_chunked(transcript: str) -> str:
    chunks = split_into_chunks(transcript)
    print(f"Transcription découpée en {len(chunks)} chunk(s)")
 
    if len(chunks) == 1:
        return generate_pv(chunks[0])
 
    partial_pvs = []
    for i, chunk in enumerate(chunks):
        print(f"PV chunk {i+1}/{len(chunks)}...")
        partial_pvs.append(generate_pv(chunk))
 
    combined = "\n\n---\n\n".join(partial_pvs)
 
    print("Fusion des PVs...")
    fusion_prompt = f"""Tu es un assistant de réunion. Voici plusieurs parties d'un procès-verbal d'une même réunion.
 
{combined}
 
Fusionne ces parties en un seul procès-verbal cohérent et chronologique, sans répétitions.
 
Procès-verbal final :"""
    return ollama_generate(fusion_prompt)
 
 
# --- Génération du résumé ---
 
def generate_summary(pv_text: str) -> str:
    prompt = f"""Tu es un assistant de réunion. Voici le procès-verbal d'une réunion.
 
PROCÈS-VERBAL :
{pv_text}
 
Génère un résumé structuré en français avec des paragraphes courts (3-4 lignes max) :

Processus obligatoire :
1. Extraire les décisions et actions potentielles.
2. Vérifier en interne que chaque élément est explicitement mentionné.
3. Supprimer tout élément basé sur une supposition, une interprétation ou une formulation ambiguë.
4. Ne PAS afficher les citations, extraits ou étapes de vérification.
5. Afficher uniquement le résultat final validé
 
Règles importantes :
- Ne rien inventer.
- Ne pas déduire des intentions implicites.
- Une décision doit être clairement validée.
- Une action doit être explicitement demandée ou assignée.
- Ne pas inventer d'action ou de décision qui n'est pas clairement mentionnée dans le procès-verbal
- Si une information est ambigue, marque la comme "peut etre faux positif IA"

## Points principaux
(un point par ligne)
 
## Décisions prises
(une décision par ligne)
 
## Actions à suivre
(une action par ligne avec le responsable si identifiable)
 
Résumé :"""
    return ollama_generate(prompt)
 
 
# --- Génération du nom de dossier ---
 
def generate_folder_name(summary: str) -> str:
    prompt = f"""À partir de ce résumé, génère un nom de dossier court (3-5 mots, en snake_case, sans accents, tout en minuscules).
 
RÉSUMÉ :
{summary[:500]}
 
Réponds uniquement avec le nom du dossier, rien d'autre."""
    return ollama_generate(prompt).strip().replace(" ", "_")
 
 
# --- Exécution ---

txt_path = paths["txt"]
transcript_text = txt_path.read_text(encoding="utf-8")
 
words = transcript_text.split()
print(f"Nombre de mots : {len(words)}")
 
print("\nGénération du procès-verbal...")
pv = generate_pv_chunked(transcript_text)
 
print("\nGénération du résumé...")
summary = generate_summary(pv)
 
folder_name = generate_folder_name(summary)
final_dir = OUTPUT_DIR / folder_name
work_dir_final = final_dir / "work"
work_dir_final.mkdir(parents=True, exist_ok=True)
 
for f in paths.values():
    shutil.move(str(f), work_dir_final / f.name)
 
(final_dir / "pv.txt").write_text(pv, encoding="utf-8")
(final_dir / "resume.txt").write_text(summary, encoding="utf-8")
 
 
print(f"\nDossier créé    : {final_dir}")
print(f"PV              → {final_dir / 'pv.txt'}")
print(f"Résumé          → {final_dir / 'resume.txt'}")
print(f"Fichiers work   → {work_dir_final}")
print("\n--- RÉSUMÉ ---")
print(summary)
print("\n--- PROCÈS-VERBAL ---")
print(pv)